In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col

bronze_df = spark.table("bronze_nba_shot_logs")
silver_df = spark.table("silver_nba_shot_logs_cleaned")
gold_df = spark.table("gold_nba_shot_quality_features")

checks = []

In [0]:
bronze_count = bronze_df.count()
silver_count = silver_df.count()
gold_count = gold_df.count()

checks.append(Row(
    check_name="bronze_to_silver_row_count",
    table_name="silver_nba_shot_logs_cleaned",
    check_type="reconciliation",
    failed_records=int(bronze_count - silver_count),
    status="PASS" if silver_count > 0 else "FAIL",
    severity="HIGH"
))

checks.append(Row(
    check_name="silver_to_gold_row_count",
    table_name="gold_nba_shot_quality_features",
    check_type="reconciliation",
    failed_records=int(silver_count - gold_count),
    status="PASS" if silver_count == gold_count else "WARN",
    severity="MEDIUM"
))

In [0]:
required_cols = [
    "shot_dist",
    "shot_clock",
    "dribbles",
    "touch_time",
    "close_def_dist",
    "period",
    "pts_type",
    "label"
]

for c in required_cols:
    null_count = gold_df.filter(col(c).isNull()).count()
    checks.append(Row(
        check_name=f"null_check_{c}",
        table_name="gold_nba_shot_quality_features",
        check_type="null_check",
        failed_records=int(null_count),
        status="PASS" if null_count == 0 else "FAIL",
        severity="HIGH"
    ))

In [0]:
range_checks = [
    ("shot_dist_negative", gold_df.filter(col("shot_dist") < 0).count()),
    ("shot_clock_negative", gold_df.filter(col("shot_clock") < 0).count()),
    ("dribbles_negative", gold_df.filter(col("dribbles") < 0).count()),
    ("touch_time_negative", gold_df.filter(col("touch_time") < 0).count()),
    ("close_def_dist_negative", gold_df.filter(col("close_def_dist") < 0).count()),
    ("invalid_label", gold_df.filter(~col("label").isin(0, 1)).count())
]

for check_name, fail_count in range_checks:
    checks.append(Row(
        check_name=check_name,
        table_name="gold_nba_shot_quality_features",
        check_type="range_check",
        failed_records=int(fail_count),
        status="PASS" if fail_count == 0 else "FAIL",
        severity="HIGH"
    ))

In [0]:
binary_feature_cols = [
    "is_three_pointer",
    "is_late_clock",
    "is_very_tight_defense",
    "is_tight_defense",
    "is_open_shot",
    "is_wide_open_shot",
    "is_close_shot",
    "is_mid_range",
    "is_deep_three"
]

for c in binary_feature_cols:
    invalid_count = gold_df.filter(~col(c).isin(0, 1)).count()
    checks.append(Row(
        check_name=f"binary_feature_check_{c}",
        table_name="gold_nba_shot_quality_features",
        check_type="binary_feature_check",
        failed_records=int(invalid_count),
        status="PASS" if invalid_count == 0 else "FAIL",
        severity="MEDIUM"
    ))

In [0]:
defense_bucket_overlap_count = gold_df.filter(
    (
        col("is_very_tight_defense") +
        col("is_tight_defense") +
        col("is_open_shot") +
        col("is_wide_open_shot")
    ) != 1
).count()

checks.append(Row(
    check_name="defender_distance_bucket_exclusivity",
    table_name="gold_nba_shot_quality_features",
    check_type="feature_logic_check",
    failed_records=int(defense_bucket_overlap_count),
    status="PASS" if defense_bucket_overlap_count == 0 else "FAIL",
    severity="HIGH"
))

In [0]:
shot_distance_bucket_overlap_count = gold_df.filter(
    (
        col("is_close_shot") +
        col("is_mid_range") +
        col("is_three_pointer")
    ) != 1
).count()

checks.append(Row(
    check_name="shot_distance_bucket_exclusivity",
    table_name="gold_nba_shot_quality_features",
    check_type="feature_logic_check",
    failed_records=int(shot_distance_bucket_overlap_count),
    status="PASS" if shot_distance_bucket_overlap_count == 0 else "FAIL",
    severity="HIGH"
))

In [0]:
make_rate = gold_df.selectExpr("avg(label) as make_rate").collect()[0]["make_rate"]

checks.append(Row(
    check_name="target_class_balance",
    table_name="gold_nba_shot_quality_features",
    check_type="ml_data_quality",
    failed_records=0,
    status="PASS" if make_rate is not None and 0.25 <= make_rate <= 0.65 else "WARN",
    severity="MEDIUM"
))

In [0]:
scorecard_df = spark.createDataFrame(checks)

scorecard_df.write.mode("overwrite").saveAsTable("gold_nba_shot_quality_scorecard")

display(scorecard_df)

In [0]:
spark.sql("""
SELECT 
  status,
  severity,
  COUNT(*) AS check_count,
  SUM(failed_records) AS total_failed_records
FROM gold_nba_shot_quality_scorecard
GROUP BY status, severity
ORDER BY severity, status
""").show()